**Archived legacy method.** Set `YEAR` and, only when intentionally writing disposable outputs, set `SAVE = True`. Outputs go to `notebooks/outputs/<YEAR>/`; this notebook does not publish canonical pipeline data.

In [ ]:
# Legacy notebook controls — edit only these values
YEAR = 2026
SAVE = False

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

RAW_ROOT = REPO_ROOT / "data" / "raw"
PARTITION_ROOT = REPO_ROOT / "data" / "partitions"
PRODUCT_ROOT = REPO_ROOT / "data" / "products"
OUTPUT_ROOT = REPO_ROOT / "notebooks" / "outputs" / str(YEAR)
GEODATA_OUTPUT_ROOT = OUTPUT_ROOT / "geodata"

DEPARTMENTS_PATH = RAW_ROOT / "demographics" / "departments.csv"
DEPARTMENT_STATS_PATH = RAW_ROOT / "demographics" / "departmental_stats_2023.csv"
ARRONDISSEMENT_STATS_PATH = RAW_ROOT / "demographics" / "arrondissement_stats_2023.csv"
DEPARTMENTS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "departments.geojson"
REGIONS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "regions.geojson"
ARRONDISSEMENTS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "arrondissements-avec-outre-mer.geojson"
PARIS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "paris_arrondissements.geojson"
MONACO_GEOMETRY_PATH = RAW_ROOT / "geodata" / "monaco.geojson"

In [1]:
def annual_restaurant_product(year):
    candidates = [
        PRODUCT_ROOT / "france" / str(year) / "all_restaurants(arrondissements).csv",
        PRODUCT_ROOT / "france" / f"all_restaurants(arrondissements)_{year % 100:02d}.csv",
        REPO_ROOT / "Years" / str(year) / "data" / "France" / "all_restaurants(arrondissements).csv",
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"No accepted annual restaurant product for {year}: {candidates}")


def save_csv(frame, filename):
    target = OUTPUT_ROOT / filename
    if not SAVE:
        print(f"SAVE=False: not writing {target}")
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(target, index=False)
    print(f"Wrote {target}")


def save_geojson(frame, filename):
    target = GEODATA_OUTPUT_ROOT / filename
    if not SAVE:
        print(f"SAVE=False: not writing {target}")
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_file(target, driver="GeoJSON")
    print(f"Wrote {target}")


# Michelin Rated Restaurants

In [2]:
import pandas as pd
pd.set_option('future.no_silent_downcasting', True)

## Initial Exploration & Preparation of Data

In [3]:
michelin = pd.read_csv(RAW_ROOT / "michelin" / f"michelin_data_{YEAR}.csv")
michelin.head(10)

,Name,Address,Location,Price,Cuisine,Longitude,Latitude,PhoneNumber,Url,WebsiteUrl,Award,GreenStar,FacilitiesAndServices,Description
0,ES:SENZ,"Mietenkamer Straße 65, Grassau, 83224, Germany","Grassau, Germany",€€€€,"Creative, Modern Cuisine",12.465618,47.785630,4.986414e+11,https://guide.michelin.com/en/bayern/grassau/r...,https://www.das-achental.com/essenz,3 Stars,0,"Air conditioning,Car park,Interesting wine list","From the get-go, the stage is set in grand sty..."
1,Tohru in der Schreiberei,"Burgstraße 5, Munich, 80331, Germany","Munich, Germany",€€€€,"Modern Cuisine, Japanese Contemporary",11.577475,48.137597,4.989215e+11,https://guide.michelin.com/en/bayern/mnchen/re...,https://schreiberei-muc.de/,3 Stars,0,"Interesting wine list,Notable sake list",It is absolutely worth climbing the 23 steps o...
2,Schwarzwaldstube,"Tonbachstraße 237, Baiersbronn, 72270, Germany","Baiersbronn, Germany",€€€€,"Classic French, Creative",8.358280,48.536911,4.974425e+11,https://guide.michelin.com/en/baden-wurttember...,https://www.traube-tonbach.de/kulinarik/schwar...,3 Stars,0,"Air conditioning,Car park,Great view,Interesti...","Schwarzwaldstube, the flagship restaurant of t..."
3,Victor's Fine Dining by christian bau,"Schlossstraße 27, Perl, 66706, Germany","Perl, Germany",€€€€,"Creative, Contemporary",6.387211,49.535173,4.968668e+10,https://guide.michelin.com/en/saarland/perl/re...,https://www.victors-fine-dining.de/,3 Stars,0,"Air conditioning,Car park,Interesting wine lis...",The way in which Christian Bau combines influe...
4,schanz. restaurant.,"Bahnhofstraße 8a, Piesport, 54498, Germany","Piesport, Germany",€€€€,"Modern French, Creative",6.926600,49.877600,4.965079e+10,https://guide.michelin.com/en/rheinland-pfalz/...,https://www.schanz-restaurant.de/de/,3 Stars,0,"Air conditioning,Car park,Wheelchair access",It is thanks to Thomas Schanz that this small ...
5,Restaurant Haerlin,"Neuer Jungfernstieg 9, Hamburg, 20354, Germany","Hamburg, Germany",€€€€,"Creative French, Classic Cuisine",9.991464,53.555442,4.940349e+08,https://guide.michelin.com/en/hamburg-region/h...,https://restaurant-haerlin.de/de/,3 Stars,0,"Air conditioning,Great view,Interesting wine l...","One of Hamburg's classic hotels, the legendary..."
6,The Table Kevin Fehling,"Shanghaiallee 15, Hamburg, 20457, Germany","Hamburg, Germany",€€€€,"Creative, International",10.003073,53.542486,4.940229e+11,https://guide.michelin.com/en/hamburg-region/h...,https://thetable-hamburg.de/,3 Stars,0,"Air conditioning,Counter dining","In his set menu (dubbed ""Das Tor zur Welt"" – ""..."
7,Waldhotel Sonnora,"Auf'm Eichelfeld 1, Dreis, 54518, Germany","Dreis, Germany",€€€€,Classic French,6.810934,49.937732,4.965790e+10,https://guide.michelin.com/en/rheinland-pfalz/...,https://www.hotel-sonnora.de/,3 Stars,0,"Car park,Garden or park,Interesting wine list",This legendary fine dining establishment has a...
8,JAN,"Luisenstraße 27, Munich, 80333, Germany","Munich, Germany",€€€€,"Creative, Modern Cuisine",11.562988,48.144798,4.989237e+11,https://guide.michelin.com/en/bayern/mnchen/re...,https://jan-hartwig.com/,3 Stars,0,Interesting wine list,This restaurant has firmly established itself ...
9,Restaurant Bareiss,"Hermine-Bareiss-Weg 1, Baiersbronn, 72270, Ger...","Baiersbronn, Germany",€€€€,"Classic French, Seasonal Cuisine",8.326825,48.520132,4.974425e+08,https://guide.michelin.com/en/baden-wurttember...,https://www.bareiss.com/,3 Stars,0,"Air conditioning,Car park,Interesting wine lis...",Claus-Peter Lumpp has been head chef at the fi...


In [4]:
print(f"Columns:\n{michelin.columns.to_list()}")

Columns:
['Name', 'Address', 'Location', 'Price', 'Cuisine', 'Longitude', 'Latitude', 'PhoneNumber', 'Url', 'WebsiteUrl', 'Award', 'GreenStar', 'FacilitiesAndServices', 'Description']


We aim to plot the coordinates on a map and search for population density correlation and compare UK and France.

We drop `Url`, `PhoneNumber` and `FacilitiesAndService`. `Url` could perhaps become useful

In [5]:
michelin = michelin[['Name', 'Address', 'Location', 'Price', 'Cuisine', 'WebsiteUrl', 'Award', 'GreenStar', 'Longitude', 'Latitude']]
michelin.head()

,Name,Address,Location,Price,Cuisine,WebsiteUrl,Award,GreenStar,Longitude,Latitude
0,ES:SENZ,"Mietenkamer Straße 65, Grassau, 83224, Germany","Grassau, Germany",€€€€,"Creative, Modern Cuisine",https://www.das-achental.com/essenz,3 Stars,0,12.465618,47.785630
1,Tohru in der Schreiberei,"Burgstraße 5, Munich, 80331, Germany","Munich, Germany",€€€€,"Modern Cuisine, Japanese Contemporary",https://schreiberei-muc.de/,3 Stars,0,11.577475,48.137597
2,Schwarzwaldstube,"Tonbachstraße 237, Baiersbronn, 72270, Germany","Baiersbronn, Germany",€€€€,"Classic French, Creative",https://www.traube-tonbach.de/kulinarik/schwar...,3 Stars,0,8.358280,48.536911
3,Victor's Fine Dining by christian bau,"Schlossstraße 27, Perl, 66706, Germany","Perl, Germany",€€€€,"Creative, Contemporary",https://www.victors-fine-dining.de/,3 Stars,0,6.387211,49.535173
4,schanz. restaurant.,"Bahnhofstraße 8a, Piesport, 54498, Germany","Piesport, Germany",€€€€,"Modern French, Creative",https://www.schanz-restaurant.de/de/,3 Stars,0,6.926600,49.877600


In [6]:
# Columns are converted to lowercase for convenience
michelin.columns = michelin.columns.str.lower()

In [7]:
michelin.rename({'websiteurl': 'url'}, axis=1, inplace=True)
print(f"Columns:\n{michelin.columns.tolist()}")

Columns:
['name', 'address', 'location', 'price', 'cuisine', 'url', 'award', 'greenstar', 'longitude', 'latitude']


In [8]:
michelin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18908 entries, 0 to 18907
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       18908 non-null  object 
 1   address    18908 non-null  object 
 2   location   18908 non-null  object 
 3   price      18908 non-null  object 
 4   cuisine    18908 non-null  object 
 5   url        16189 non-null  object 
 6   award      18908 non-null  object 
 7   greenstar  18908 non-null  int64  
 8   longitude  18908 non-null  float64
 9   latitude   18908 non-null  float64
dtypes: float64(2), int64(1), object(7)
memory usage: 1.4+ MB


There exist missing values which will be dealt with once partitioned by location

Range of values for specific columns

In [9]:
ignore = ['longitude', 'latitude', 'url', 'cuisine']

for column in michelin:
    if column in ignore:
        pass
    else:
        print(f"\nUnique {column}s: {michelin[column].unique()}\nTotal Unique: {len(michelin[column].unique())} values")


Unique names: ['ES:SENZ' 'Tohru in der Schreiberei' 'Schwarzwaldstube' ... 'Gustificio'
 'Inka' 'Casa Celesia']
Total Unique: 18213 values

Unique addresss: ['Mietenkamer Straße 65, Grassau, 83224, Germany'
 'Burgstraße 5, Munich, 80331, Germany'
 'Tonbachstraße 237, Baiersbronn, 72270, Germany' ...
 'Via Guglielmo Marconi 36, Carmignano di Brenta, 35010, Italy'
 'via Guglielmo Oberdan 18/A, Verona, 37121, Italy'
 'Piazza Martiri della Libertà, 31, Oleggio, 28047, Italy']
Total Unique: 18451 values

Unique locations: ['Grassau, Germany' 'Munich, Germany' 'Baiersbronn, Germany' ...
 'Ponte de Lima, Portugal' 'Carmignano di Brenta, Italy' 'Oleggio, Italy']
Total Unique: 6013 values

Unique prices: ['€€€€' '$$$$' '€€€' '€€' '¥¥¥¥' '¥¥¥' '₫₫₫₫' '₫₫₫' '₫₫' '$' '$$$' '$$'
 '¥¥' '€' '₫' '¥' '฿฿' '฿฿฿' '฿' '₺₺' '₺₺₺' '฿฿฿฿' '₺₺₺₺' '﷼﷼﷼﷼' '﷼﷼﷼' '﷼'
 '﷼﷼' '₺' '₩₩₩' '₩' '₩₩' '£££' '££££' '££' '£' '₩₩₩₩']
Total Unique: 36 values

Unique awards: ['3 Stars' 'Selected Restaurants' '1 Star' '2 Stars'

----
&nbsp;
## Separate `location` column

- `country` column. eg, 'USA'
- `city` column. eg, 'San Fransisco'

`location` is comma separated "country, city"

In [10]:
# Are there non-comma separated entries in `location`?
no_comma = michelin[~michelin['location'].str.contains(',')]
no_comma['location'].unique().tolist()

['Dubai', 'Luxembourg', 'Singapore', 'Abu Dhabi', 'Macau']

These are all 'city states' or principalities.

In [11]:
# Create a dictionary for special cases
special_cases = {'Hong Kong': 'Hong Kong, Hong Kong SAR China',
                 'Macau': 'Macau, Macau SAR China',
                 'Singapore': 'Singapore, Singapore',
                 'Dubai': 'Dubai, United Arab Emirates',
                 'Luxembourg': 'Luxembourg, Luxembourg',
                 'Abu Dhabi': 'Abu Dhabi, United Arab Emirates'}

US restaurants have been reformated

In [12]:
problem_locations = (
    michelin['location']
    .astype('string')
    [michelin['location'].astype('string').str.count(',') > 1]
    .dropna()
    .unique()
)

print(problem_locations)
print(f"Unique location values with more than one comma: {len(problem_locations)}")

<StringArray>
[           'Miami, FL, USA',          'Orlando, FL, USA',
  'Fort Lauderdale, FL, USA',            'Tampa, FL, USA',
      'Miami Beach, FL, USA',     'Coral Gables, FL, USA',
      'Winter Park, FL, USA',         'Surfside, FL, USA',
  'West Palm Beach, FL, USA',    'Winter Garden, FL, USA',
 ...
            'Bronx, NY, USA',        'Peekskill, NY, USA',
 'Long Island City, NY, USA',     'New Rochelle, NY, USA',
      'Mount Kisco, NY, USA',          'Astoria, NY, USA',
              'Rye, NY, USA',        'Hartsdale, NY, USA',
       'Mamaroneck, NY, USA',    'West Harrison, NY, USA']
Length: 244, dtype: string
Unique location values with more than one comma: 244


In [13]:
# Apply special cases first
michelin['location'] = michelin['location'].replace(special_cases)

# Clean and normalize
loc = michelin['location'].str.strip()

# Split from the right into at most 3 parts:
# city | state | country
parts = loc.str.rsplit(',', n=2, expand=True)

# Guarantee 3 columns
for i in range(parts.shape[1], 3):
    parts[i] = pd.NA

# If only 2 parts exist, treat them as city + country
if parts.shape[1] >= 2:
    two_part_rows = parts[2].isna()
    parts.loc[two_part_rows, 2] = parts.loc[two_part_rows, 1]
    parts.loc[two_part_rows, 1] = pd.NA

parts.columns = ['city', 'state', 'country']

# Trim whitespace
for col in ['city', 'state', 'country']:
    parts[col] = parts[col].astype('string').str.strip()

# Replace original column
michelin = michelin.drop(columns=['location']).join(parts)

Legacy below

In [14]:
# # Apply special cases to the 'location' column
# michelin['location'] = michelin['location'].replace(special_cases)
#
# # Now split the 'location' column
# locations = michelin['location'].str.split(',', expand=True)
# locations.columns = ['city', 'country']
#
# # Remove leading or trailing whitespace from 'city' and 'country' columns
# locations['city'] = locations['city'].str.strip()
# locations['country'] = locations['country'].str.strip()
#
# # Replace the original 'location' column with the new 'country' and 'city' columns
# michelin = michelin.drop('location', axis=1).join(locations)

In [15]:
print(f"New Columns: {michelin.columns.tolist()}")

New Columns: ['name', 'address', 'price', 'cuisine', 'url', 'award', 'greenstar', 'longitude', 'latitude', 'city', 'state', 'country']


In [16]:
print(f"Unique Countries: {michelin['country'].unique()}"
      f"\nTotal Unique = {len(michelin['country'].unique())} values")

Unique Countries: <StringArray>
[               'Germany',                 'Norway',                 'Sweden',
                'Denmark',               'Slovenia',   'United Arab Emirates',
                'Belgium',                  'Japan',                'Finland',
                 'Poland',                 'Mexico',                 'Canada',
                 'Brazil',       'Chinese Mainland',             'Luxembourg',
                    'USA',              'Argentina',                 'Greece',
                'Iceland',                'Vietnam',              'Lithuania',
                'Estonia',                 'Taiwan',                  'Italy',
                  'Spain',            'Switzerland',              'Singapore',
               'Thailand',            'Netherlands',                'Croatia',
                'Türkiye',        'The Philippines',                'Hungary',
                 'Latvia',          'Liechtenstein',           'Saudi Arabia',
               'Mala

In [17]:
print(f"Unique Cities: {michelin['city'].unique()}"
      f"\nTotal Unique = {len(michelin['city'].unique())} values")

Unique Cities: <StringArray>
[             'Grassau',               'Munich',          'Baiersbronn',
                 'Perl',             'Piesport',              'Hamburg',
                'Dreis',               'Berlin',                 'Oslo',
            'Stavanger',
 ...
              'Portela',            'Quarteira',           'Cantanhede',
                'Loulé',             'Odeceixe',   'Moreira de Cónegos',
           'Matosinhos',        'Ponte de Lima', 'Carmignano di Brenta',
              'Oleggio']
Length: 5993, dtype: string
Total Unique = 5993 values


In [18]:
michelin = michelin[['name', 'address', 'city', 'country', 'price', 'cuisine', 'url', 'award', 'greenstar', 'longitude', 'latitude']]
michelin.head()

,name,address,city,country,price,cuisine,url,award,greenstar,longitude,latitude
0,ES:SENZ,"Mietenkamer Straße 65, Grassau, 83224, Germany",Grassau,Germany,€€€€,"Creative, Modern Cuisine",https://www.das-achental.com/essenz,3 Stars,0,12.465618,47.785630
1,Tohru in der Schreiberei,"Burgstraße 5, Munich, 80331, Germany",Munich,Germany,€€€€,"Modern Cuisine, Japanese Contemporary",https://schreiberei-muc.de/,3 Stars,0,11.577475,48.137597
2,Schwarzwaldstube,"Tonbachstraße 237, Baiersbronn, 72270, Germany",Baiersbronn,Germany,€€€€,"Classic French, Creative",https://www.traube-tonbach.de/kulinarik/schwar...,3 Stars,0,8.358280,48.536911
3,Victor's Fine Dining by christian bau,"Schlossstraße 27, Perl, 66706, Germany",Perl,Germany,€€€€,"Creative, Contemporary",https://www.victors-fine-dining.de/,3 Stars,0,6.387211,49.535173
4,schanz. restaurant.,"Bahnhofstraße 8a, Piesport, 54498, Germany",Piesport,Germany,€€€€,"Modern French, Creative",https://www.schanz-restaurant.de/de/,3 Stars,0,6.926600,49.877600


-----
&nbsp;
### Count by country

In [19]:
pivot_table = michelin.pivot_table(index='country', columns=['award'], aggfunc='size', fill_value=0)
pivot_table

award,1 Star,2 Stars,3 Stars,Bib Gourmand,Selected Restaurants
country,,,,,
Andorra,1,0,0,0,5
Argentina,9,1,0,10,54
Austria,79,19,2,61,258
Belgium,104,18,2,118,501
Brazil,19,5,0,39,77
Canada,36,2,0,58,183
Chinese Mainland,117,21,3,252,268
Croatia,12,1,0,12,73
Czechia,8,1,0,18,52


In [20]:
michelin_filtered = michelin[michelin['award'] != 'Bib Gourmand']
pivot_table_filtered = michelin_filtered.pivot_table(index='country', columns='award', aggfunc='size', fill_value=0)
pivot_table_filtered['total_ratings'] = pivot_table_filtered.sum(axis=1)

# Rank the countries based on the total ratings
pivot_table_filtered['rank'] = pivot_table_filtered['total_ratings'].rank(ascending=False)
pivot_table_sorted = pivot_table_filtered.sort_values(by='rank')
pivot_table_sorted

award,1 Star,2 Stars,3 Stars,Selected Restaurants,total_ratings,rank
country,,,,,,
France,547,81,30,1967,2625,1.0
Italy,335,38,15,1321,1709,2.0
USA,222,40,14,1053,1329,3.0
Spain,251,37,16,782,1086,4.0
Germany,273,45,11,749,1078,5.0
United Kingdom,171,23,10,756,960,6.0
Japan,274,57,19,530,880,7.0
Belgium,104,18,2,501,625,8.0
Switzerland,108,27,4,294,433,9.0


In [21]:
# Filter out 'Bib Gourmand'
michelin_filtered = michelin[michelin['award'] != 'Bib Gourmand']

# Create a pivot table for the total number of star ratings
star_rating_pivot = michelin_filtered.pivot_table(index='award', aggfunc='size', fill_value=0)
star_rating_pivot.loc['Total'] = star_rating_pivot.sum()
star_rating_pivot

award
1 Star                   3145
2 Stars                   536
3 Stars                   154
Selected Restaurants    11507
Total                   15342
dtype: int64

----

## `awards` columns

In the Michelin dataset, we have a column named 'awards' that designates the level of recognition a restaurant has achieved according to Michelin's rating system. These awards are '3 Stars', '2 Stars', '1 Star', and 'Bib Gourmand', which is a different award for good quality, good value restaurants.

However, in order to make the analysis more tractable and to create a more uniform scale, we transform these awards into numerical values. This transformation will allow us to perform quantitative analysis and make mathematical computations with this data, which wouldn't be possible with the original textual data.

The '3 MICHELIN Stars', '2 MICHELIN Stars', and '1 MICHELIN Star' awards are straightforwardly transformed into the numerical values 3, 2, and 1, respectively. However, the 'Bib Gourmand' award doesn't fit directly into this star system. In consideration of the prestige and value attached to this award, we've decided to map 'Bib Gourmand' to the value 0.5. It's important to note that this decision, while somewhat arbitrary, is made with the understanding that the 'Bib Gourmand' recognizes a different aspect of restaurant quality and is not strictly comparable to the star awards.

In [22]:
award_dict = {'3 Stars': 3, '2 Stars': 2, '1 Star': 1, 'Bib Gourmand': 0.5, 'Selected Restaurants': 0.25}
michelin['stars'] = michelin['award'].replace(award_dict).infer_objects(copy=False).astype(float)

In [23]:
cols = michelin.columns.tolist()

cols.remove('stars')
# insert 'stars' at the desired position next to 'award' which we retain
cols.insert(-2, 'stars')

# reindex the DataFrame
michelin = michelin.reindex(columns=cols)

In [24]:
michelin.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18908 entries, 0 to 18907
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       18908 non-null  object 
 1   address    18908 non-null  object 
 2   city       18908 non-null  string 
 3   country    18908 non-null  string 
 4   price      18908 non-null  object 
 5   cuisine    18908 non-null  object 
 6   url        16189 non-null  object 
 7   award      18908 non-null  object 
 8   greenstar  18908 non-null  int64  
 9   stars      18908 non-null  float64
 10  longitude  18908 non-null  float64
 11  latitude   18908 non-null  float64
dtypes: float64(3), int64(1), object(6), string(2)
memory usage: 1.7+ MB


----
&nbsp;
## `price` column

There is also the price column to organise which lists a number of different currencies. It is unclear if, for example, $ refers to USD, HKD etc..

This attribute is easier to deal with piecewise by country

----
&nbsp;
## Export `UK` and `France` datasets for further analysis

In [25]:
# Filter the DataFrame for records where country == 'UK'
uk_data = michelin[michelin['country'] == 'United Kingdom']
uk_data.head()

,name,address,city,country,price,cuisine,url,award,greenstar,stars,longitude,latitude
13287,74 Charlotte Street by Ben Murphy,"74 Charlotte Street, Fitzrovia, London, W1T 4Q...",London,United Kingdom,£££,Modern British,https://74charlottestreet.com/,Selected Restaurants,0,0.25,-0.136629,51.520543
13288,Pignut & The Hare,"Hare inn, Scawton, YO7 2HG, United Kingdom",Scawton,United Kingdom,££££,Creative,https://www.pignutandthehare.co.uk,Selected Restaurants,0,0.25,-1.159247,54.243391
13289,Vinette,"36 Broughton Street, Edinburgh, EH1 3SB, Unite...",Edinburgh,United Kingdom,££,Modern British,http://www.vinette.co.uk/,Selected Restaurants,0,0.25,-3.189809,55.958278
13290,Restaurant FIR,"The Vine Tree, Legar Road, Llangattock, NP8 1H...",Llangattock,United Kingdom,££££,Modern Cuisine,https://www.restaurantfir.com/,Selected Restaurants,0,0.25,-3.142319,51.855309
13291,Cicoria,"5th Floor, Royal Opera House, Bow Street, Cove...",London,United Kingdom,£££,Italian,https://www.rbo.org.uk/visit/eat-and-drink/cic...,Selected Restaurants,0,0.25,-0.122956,51.513479


In [26]:
# Export the UK data to a csv file
save_csv(uk_data, f"uk_{YEAR}.csv")

SAVE=False: not writing /Users/Ian/Documents/Study/Stage3/Programming/Github/Michelin_Rated_Restaurants/notebooks/outputs/2026/uk_2026.csv


In [27]:
# Filter the DataFrame for records where country == 'France'
france_data = michelin[michelin['country'] == 'France']
france_data.head()

,name,address,city,country,price,cuisine,url,award,greenstar,stars,longitude,latitude
13325,Le Cercle de Montaigne,"144 rue Abbé-de-l'Epée, Bordeaux, 33000, France",Bordeaux,France,€€€,Modern Cuisine,https://www.hotel-palais-gallien-bordeaux.com/...,Selected Restaurants,0,0.25,-0.584132,44.846077
13326,Épopée,"52 rue Léon-Frot, Paris, 75011, France",Paris,France,€€,Modern Cuisine,https://www.epopee-charonne.fr,Bib Gourmand,0,0.50,2.386164,48.856151
13327,La Table 1625,"391 route de la Charvaz, Jongieux, 73170, France",Jongieux,France,€€,Traditional Cuisine,https://www.chateaudelamar.fr/restaurant-la-ta...,Selected Restaurants,0,0.25,5.796453,45.742677
13328,L'Épicurieux chez Luc,"1080 route des Monts-d'Or, Curis-au-Mont-d'Or,...",Curis-au-Mont-d'Or,France,€,Traditional Cuisine,https://www.lepicurieuxchezluc.fr,Bib Gourmand,0,0.50,4.820081,45.870098
13329,Vertfeuille,"9 place de l’Eglise, Arc-et-Senans, 25610, France",Arc-et-Senans,France,€€,Modern Cuisine,https://www.vertfeuille-restaurant.com,Bib Gourmand,0,0.50,5.788186,47.036622


In [28]:
# Define and export Monaco
monaco = michelin[michelin['country'] == 'Principality of Monaco']
save_csv(monaco, f"monaco_{YEAR}.csv")

SAVE=False: not writing /Users/Ian/Documents/Study/Stage3/Programming/Github/Michelin_Rated_Restaurants/notebooks/outputs/2026/monaco_2026.csv


We remove Monaco from metropolitan France

In [29]:
france_data = france_data[france_data['city'] != 'Monaco']
france_data.shape

(3055, 12)

In [30]:
# Export the France data to a csv file
save_csv(france_data, f"france_{YEAR}.csv")

SAVE=False: not writing /Users/Ian/Documents/Study/Stage3/Programming/Github/Michelin_Rated_Restaurants/notebooks/outputs/2026/france_2026.csv


----
&nbsp;
## Restaurants could be further partitioned by country from this point